In [1]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from pybamm import exp
from pybamm import tanh

Defining Model Variables

In [2]:

xi = pybamm.SpatialVariable(
    "xi", domain="SEI layer", coord_sys="cartesian")

CL_atom = pybamm.Variable(
    "concentration of netutral lithium atoms in the SEI [mol.m-3]",  domain="SEI layer")
CL_ion = pybamm.Variable(
    "concentration of llithium ions in the SEI [mol.m-3]",  domain="SEI layer")
Phi_SEI = pybamm.Variable("Potential in the SEI [V]",  domain="SEI layer")
L_SEI = pybamm.Variable("Thickness of SEI [m]")

In [3]:
model = pybamm.lithium_ion.BaseModel()

Defining parameters of the model

In [4]:
T = pybamm.Parameter('Initial temperature [K]')
R = pybamm.Parameter('Ideal gas constant [J.K-1.mol-1]')
F = pybamm.Parameter("Faraday constant [C.mol-1]")
D_Li_atom = pybamm.Parameter(
    'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]')
D_Li_ion = pybamm.Parameter(
    'diffusion coefficient of llithium ions in the SEI [m2.s-1]')

CL_atom_0 = pybamm.Parameter(
    "initial concentration of netutral lithium atoms in the SEI [mol.m-3]")
CL_ion_0 = pybamm.Parameter(
    "initial concentration of llithium ions in the SEI [mol.m-3]")

Phi_SEI_0 = pybamm.Parameter("initial potential in the SEI [V]")


M_SEI = pybamm.Parameter("molar weight of SEI material [kg.mol-1]")
rho_SEI = pybamm.Parameter("density of SEI material [kg.m-3]")
n_SEI = pybamm.Parameter("electrone number [-]")


L_tun = pybamm.Parameter("length of the tunneling [m]")
L_SEI_0 = pybamm.Parameter("initial thickness of SEI [m]")

CL_ion_max = pybamm.Parameter(
    "maximum concentration of llithium ions in the electrolyte [mol.m-3]")

j0_0 = pybamm.Parameter(
    "Butler-Volmer rate constant for intercalation [A.m-2]")

J_total = pybamm.Parameter("Total current density [A.m-2]")

In [5]:
param = pybamm.ParameterValues(
    {
        'Initial temperature [K]': 298.15,
        'Ideal gas constant [J.K-1.mol-1]': 8.314462618,
        "Faraday constant [C.mol-1]": 96485.33212,
        'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]': 1,
        'diffusion coefficient of llithium ions in the SEI [m2.s-1]': 29866.0,
        'initial concentration of netutral lithium atoms in the SEI [mol.m-3]': 0,
        'initial concentration of llithium ions in the SEI [mol.m-3]': 1,
        'initial potential in the SEI [V]': 0,
        'molar weight of SEI material [kg.mol-1]': 0.162,
        'density of SEI material [kg.m-3]': 1690,
        'electrone number [-]': 2,
        'length of the tunneling [m]': 2e-9,
        'initial thickness of SEI [m]': 1e-9,
        'maximum concentration of llithium ions in the electrolyte [mol.m-3]': 1000,
        'Butler-Volmer rate constant for intercalation [A.m-2]': 6.4e-7,
        "Total current density [A.m-2]": 0.1


    }
)

In [6]:
NL_ions = D_Li_ion / L_SEI * pybamm.grad(CL_ion) - D_Li_ion * F/(
    R*T * L_SEI)*pybamm.grad(Phi_SEI)  # define the flux for lithoum ions
# define the flux for netutral lithium atoms
NL_atom = D_Li_atom / L_SEI * pybamm.grad(CL_atom)

V_SEI = M_SEI/(n_SEI*rho_SEI)

# Unclear how to define the source terms
A = 1
R_CLi_ions = 0
R_CLi_atom = 0


# J_Li_0 = NL_atom - D_Li_atom*F/(R*T * L_SEI)*CL_atom*pybamm.grad(Phi_SEI)
# J_tun = A * J_Li_0 * pybamm.exp(- L_SEI / L_tun)
# Je = J_Li_0 + J_tun
alpha = 0.2
j_sei_0 = 7.04e-5
J_sei = j_sei_0 * (pybamm.exp((1-alpha)*F/(R*T)*Phi_SEI) -
                   pybamm.exp(-alpha*F/(R*T)*Phi_SEI))

# define the rhs equation
# dL_SEI_dt = - V_SEI / F * pybamm.BoundaryValue(J_sei, "right")
# dCL_ion_dt = 1/L_SEI * dL_SEI_dt * \
#     pybamm.inner(xi, pybamm.grad(CL_ion)) - 1 / L_SEI * pybamm.div(NL_ions) + \
#     R_CLi_ions  # define the rhs equation
# dCL_atom_dt = 1/L_SEI * dL_SEI_dt * \
#     pybamm.inner(xi, pybamm.grad(CL_ion))-1/L_SEI * pybamm.div(NL_atom) + \
#     R_CLi_atom  # define the rhs equation


dL_SEI_dt = - V_SEI / F * 1
dCL_ion_dt = pybamm.div(NL_ions) + \
    R_CLi_ions  # define the rhs equation
dCL_atom_dt = pybamm.div(NL_atom) + \
    R_CLi_atom  # define the rhs equation

In [7]:

eta_int = Phi_SEI - Phi_SEI_0
J_int = 2*j0_0 * pybamm.sinh(F/(2*R*T) * Phi_SEI)  # Prescribed
model.algebraic = {Phi_SEI: J_total - (J_int+J_sei)}
model.rhs = {CL_ion: dCL_ion_dt, CL_atom: dCL_atom_dt, L_SEI: dL_SEI_dt}

In [8]:
model.variables = {
    "concentration of netutral lithium atoms in the SEI [mol.m-3]": CL_atom,
    "concentration of llithium ions in the SEI [mol.m-3]": CL_ion,
    "Potential in the SEI [V]": Phi_SEI,
    "Thickness of SEI [m]": L_SEI
}

In [9]:
model.initial_conditions = {CL_ion: CL_ion_0, CL_atom: CL_atom_0,
                            Phi_SEI: Phi_SEI_0, L_SEI: L_SEI_0}

lbc_CL_ion = pybamm.BoundaryValue(J_int, "left")/(F*D_Li_ion)
rbc_CL_ion = CL_ion_max

lbc_CL_atom = pybamm.BoundaryValue(J_sei, "left")/(F*D_Li_atom)
rbc_CL_atom = 0

model.boundary_conditions = {CL_ion: {"left": (lbc_CL_ion, "Neumann"), "right": (rbc_CL_ion, "Dirichlet")},
                             CL_atom: {"left": (lbc_CL_atom, "Neumann"), "right": (rbc_CL_atom, "Dirichlet")}}
# model.boundary_conditions = {c: {"left": (lbc, "Neumann"), "right": (rbc, "Neumann")}}

In [10]:

geometry = pybamm.Geometry({"SEI layer": {xi: {"min": 0, "max": 1}}})

In [11]:
param.process_model(model)
param.process_geometry(geometry)
submesh_types = {"SEI layer": pybamm.Uniform1DSubMesh}
var_pts = {xi: 50}
# # create a mesh of our geometry, using a uniform grid with 20 volumes
mesh = pybamm.Mesh(geometry, submesh_types, var_pts)
spatial_methods = {"SEI layer": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)
disc.process_model(model)

ShapeError: Cannot find shape (original error: operands could not be broadcast together with shapes (51,1) (49,1) )

In [ ]:
# solver = pybamm.ScipySolver()
# pybamm.CasadiSolver(mode="fast")
solver = pybamm.IDAKLUSolver()

In [ ]:
sim = pybamm.Simulation(
    model,
    geometry=geometry,
    parameter_values=param,
    var_pts=var_pts,
    spatial_methods=spatial_methods,
    solver=solver,
)

sim.solve([0, 1])